# Phase 06B.04 — Confirmatory statistics and winner lock

Exact schema-v2 registry, two preregistered comparison families, 10,000 paired group-cluster resamples, exact McNemar and within-family Holm correction.

In [ ]:
from pathlib import Path
import json,sys
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/"src").is_dir()),None)
if ROOT is None: raise RuntimeError("Run inside RoadBuddy")
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))
from roadbuddy_common import save_json
from phase06a_common import compute_classification_metrics,exact_mcnemar,holm_adjust,paired_group_cluster_bootstrap,sha256_file,sha256_json,validate_prediction_artifact
from phase06b_common import EXPERIMENT_ROLES,LOCKED_BASELINE_WINNER,WINNER_ELIGIBLE_ARMS,validate_full_temporal_predictions,validate_locked_baseline,validate_temporal_experiment_registry
P=ROOT/"outputs/phase06b/novelty_protocol/novelty_protocol.json"; R=ROOT/"outputs/phase06b/experiments/novelty_experiment_registry.json"
OUT=ROOT/"outputs/phase06b/final_analysis"; OUT.mkdir(parents=True,exist_ok=True); IDS=ROOT/"data/splits/phase01/validation_sample_ids.json"
baseline_manifest=validate_locked_baseline(ROOT)
if not P.is_file(): raise RuntimeError("Novelty protocol is not locked by a human decision")
protocol=json.loads(P.read_text())


In [ ]:
if not R.is_file():
    arms=[]
    for name,(role,selector,k,eligible) in EXPERIMENT_ROLES.items():
        arms.append({"name":name,"experiment_role":role,"selector_family":selector,"k":k,"candidate_count":32,"total_visual_tile_budget":8,"tile_allocation":[8] if k==1 else ([3,2,3] if k==3 else [1]*8),"config_sha256":"","checkpoint_sha256":"","feature_bank_sha256":"","predictions_path":"","predictions_sha256":"","validation_ids_sha256":"dbfa2d337f56bd681df70206fa8cee83d743c6743a3493619083d03b804145c5","winner_eligible":eligible})
    template={"schema_version":2,"status":"template_only","selected_track":"traffic_temporal_grounding","parent_protocol_sha256":sha256_json(protocol),"validation_ids_sha256":"dbfa2d337f56bd681df70206fa8cee83d743c6743a3493619083d03b804145c5","arms":arms,"winner_candidates":sorted(WINNER_ELIGIBLE_ARMS)}
    save_json(ROOT/"outputs/phase06b/experiments/novelty_experiment_registry.template.json",template)
    save_json(OUT/"PHASE06B_04_STATUS.json",{"status":"awaiting_experiments"})
    raise RuntimeError("Lock the exact experiment registry and provide every full arm")
registry=validate_temporal_experiment_registry(json.loads(R.read_text()),protocol=protocol)
ids=json.loads(IDS.read_text()); by_name={arm["name"]:arm for arm in registry["arms"]}


In [ ]:
def load_arm(name):
    arm=by_name[name]; path=Path(arm["predictions_path"]); path=path if path.is_absolute() else ROOT/path
    if not path.is_file() or sha256_file(path)!=arm["predictions_sha256"]: raise ValueError(f"Artifact hash mismatch: {name}")
    frame=pd.read_csv(path)
    if name==LOCKED_BASELINE_WINNER: validate_prediction_artifact(frame,ids,run_scope="full")
    else: validate_full_temporal_predictions(frame,ids,arm=arm)
    return frame
frames={name:load_arm(name) for name in by_name}
control=frames[LOCKED_BASELINE_WINNER]
leader=[]; per_class=[]; efficiency=[]
for name,frame in frames.items():
    metric=compute_classification_metrics(frame); leader.append({"experiment":name,"accuracy":metric["accuracy"],"macro_f1":metric["macro_f1"],"parse_rate":metric["parse_rate"]})
    for label,values in metric["per_class"].items(): per_class.append({"experiment":name,"class":label,**values})
    efficiency.append({"experiment":name,"feature_latency_seconds":float(frame.get("feature_latency_seconds",pd.Series([0.0])).mean()),"selector_latency_seconds":float(frame.get("selector_latency_seconds",pd.Series([0.0])).mean()),"vlm_latency_seconds":float(frame.get("vlm_latency_seconds",frame.get("latency_seconds",pd.Series([0.0]))).mean()),"mean_realized_tiles":float(frame.realized_tile_count.mean())})
pd.DataFrame(leader).to_csv(OUT/"novelty_leaderboard.csv",index=False); pd.DataFrame(per_class).to_csv(OUT/"per_class_metrics.csv",index=False); pd.DataFrame(efficiency).to_csv(OUT/"efficiency_table.csv",index=False)


In [ ]:
def compare(left_name,right_name,seed):
    left,right=frames[left_name],frames[right_name]
    dist,summary=paired_group_cluster_bootstrap(left,right,resamples=10_000,seed=seed)
    dist.to_csv(OUT/f"bootstrap_{right_name}_vs_{left_name}.csv",index=False)
    l=left.sort_values("sample_id").correct.astype(bool); r=right.sort_values("sample_id").correct.astype(bool)
    test=exact_mcnemar(l,r); d=summary["accuracy_delta"]
    transitions=pd.DataFrame({"sample_id":left.sort_values("sample_id").sample_id,"left":l.values,"right":r.values,"comparison":f"{right_name}_vs_{left_name}"})
    return {"comparison":f"{right_name}_vs_{left_name}","left":left_name,"right":right_name,"accuracy_delta_mean":d["mean"],"ci95_low":d["ci95_low"],"ci95_high":d["ci95_high"],"mcnemar_raw_p":test["exact_two_sided_p"]},transitions
family1=[]; family2=[]; transitions=[]
for i,name in enumerate(sorted(WINNER_ELIGIBLE_ARMS-{"L32-F1"})):
    row,tr=compare("L32-F1",name,42+i); family1.append(row); transitions.append(tr)
for i,k in enumerate([1,3,8]):
    row,tr=compare(f"QTG-{k}",f"TATG-{k}",142+i); family2.append(row); transitions.append(tr)
for family in [family1,family2]:
    adjusted=holm_adjust({row["comparison"]:row["mcnemar_raw_p"] for row in family})
    for row in family: row["mcnemar_holm_p"]=adjusted[row["comparison"]]
stats=pd.DataFrame([{**row,"family":"novelty_vs_baseline"} for row in family1]+[{**row,"family":"traffic_contribution"} for row in family2])
stats["qualifies"]=False
stats.loc[stats.family.eq("novelty_vs_baseline"),"qualifies"]=(stats.ci95_low>0)&(stats.mcnemar_holm_p<.05)
stats.to_csv(OUT/"mcnemar_holm_tables.csv",index=False); pd.concat(transitions).to_csv(OUT/"paired_transitions.csv",index=False)


In [ ]:
eligible=stats[(stats.family=="novelty_vs_baseline") & stats.qualifies].sort_values("accuracy_delta_mean",ascending=False)
winner=LOCKED_BASELINE_WINNER if eligible.empty else eligible.iloc[0].right
reason="No novelty arm passed both confirmatory gates; retain L32-F1." if eligible.empty else "Highest preregistered accuracy delta among arms passing both gates."
manifest={"schema_version":2,"status":"complete","winner":winner,"baseline_control":LOCKED_BASELINE_WINNER,"selected_track":protocol["selected_track"],"decision":reason,"protocol_sha256":sha256_json(protocol),"registry_sha256":sha256_file(R),"validation_rows":len(ids),"validation_unique_ids":len(set(map(str,ids))),"bootstrap_resamples":10000,"holm_alpha":.05,"oracle_in_winner_registry":False,"public_test_accessed":False}
save_json(OUT/"novelty_winner_manifest.json",manifest); save_json(OUT/"PHASE06B_04_STATUS.json",{"status":"complete","winner":winner,"validation_rows":298})
(OUT/"PHASE06B_REPRODUCIBILITY_REPORT.md").write_text(f"# Phase 06B reproducibility report\n\nWinner: **{winner}**. {reason}\n\nAll comparisons use 10,000 paired group-cluster resamples. Exact McNemar is secondary and Holm-adjusted within preregistered families. Non-significance is not equivalence. Oracle is diagnostic-only. No public-test artifact was accessed or created.\n")
manifest
